In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

# Último día cubierto por Analytics (rango fijado en el checkpoint)
ANALYTICS_END = pd.Timestamp("2026-10-20", tz="UTC")

perf = pd.read_csv(CSV_DIR / "video_performance.csv")
perf["published_at"] = pd.to_datetime(perf["published_at"], utc=True)

# --- 6.1 Base de comparación: solo vídeos que tienen datos de Analytics -----
df = perf[perf["views"].notna()].copy()
print(f"== 6.1 Vídeos comparables: {len(df)} de {len(perf)} ==")

# Diferencia absoluta y relativa (base: Analytics). Si views = 0, el % no se define.
df["diff_views"] = df["view_count"] - df["views"]
base = df["views"].where(df["views"] > 0)
df["diff_views_pct"] = df["diff_views"] / base * 100
df["dias_desde_publicacion"] = (ANALYTICS_END - df["published_at"]).dt.days

sin_base = int(df["diff_views_pct"].isna().sum())
print(f"Vídeos con views = 0 en Analytics (sin % calculable): {sin_base}")

# --- 6.2 Totales y signo de la diferencia -----------------------------------
total_dc, total_an = df["view_count"].sum(), df["views"].sum()
print("\n== 6.2 Totales ==")
print(f"Data API (view_count): {total_dc:,.0f}")
print(f"Analytics (views):     {total_an:,.0f}")
print(f"Diferencia: {total_dc - total_an:,.0f} ({(total_dc - total_an) / total_an:.2%})")

signo = np.sign(df["diff_views"]).map(
    {1.0: "Data API > Analytics", 0.0: "iguales", -1.0: "Data API < Analytics"}
)
print("\nVídeos según el signo de la diferencia:")
print(signo.value_counts().to_string())

# --- 6.3 Distribución de la diferencia relativa -----------------------------
tramos = pd.cut(
    df["diff_views_pct"].abs(),
    bins=[-0.001, 0, 1, 5, np.inf],
    labels=["exacta (0%)", "hasta 1%", "1% - 5%", "más de 5%"],
)
print("\n== 6.3 Vídeos por tramo de diferencia relativa (valor absoluto) ==")
print(tramos.value_counts(sort=False).to_string())
print("\nResumen de diff_views_pct (%):")
print(df["diff_views_pct"].describe().round(2).to_string())

# --- 6.4 Los 15 vídeos con mayor diferencia absoluta ------------------------
columnas = [
    "video_id", "title", "is_public", "format", "published_at",
    "view_count", "views", "diff_views", "diff_views_pct",
]
top = df.reindex(df["diff_views"].abs().sort_values(ascending=False).index).head(15)
print("\n== 6.4 Mayores diferencias absolutas ==")
print(top[columnas].round(1).to_string(index=False))

# --- 6.5 Diferencia por antigüedad y por formato ----------------------------
df["antiguedad"] = pd.cut(
    df["dias_desde_publicacion"],
    bins=[-np.inf, 7, 30, 90, 365, np.inf],
    labels=["hasta 7 d", "8-30 d", "31-90 d", "91-365 d", "más de 365 d"],
)


def resumen_diferencia(grupo: pd.core.groupby.DataFrameGroupBy) -> pd.DataFrame:
    """Resume una agrupación: nº de vídeos, totales y diferencia relativa."""
    out = grupo.agg(
        videos=("video_id", "count"),
        data_api=("view_count", "sum"),
        analytics=("views", "sum"),
        mediana_pct=("diff_views_pct", "median"),
    )
    out["diff_total_%"] = (out["data_api"] - out["analytics"]) / out["analytics"] * 100
    return out.round(2)


print("\n== 6.5a Por antigüedad (días entre publicación y 2026-10-20) ==")
print(resumen_diferencia(df.groupby("antiguedad", observed=True)).to_string())

print("\n== 6.5b Por formato ==")
print(resumen_diferencia(df.groupby("format")).to_string())

# --- 6.6 Likes y comentarios: totales de ambas fuentes ----------------------
print("\n== 6.6 Likes y comentarios (totales) ==")
for etiqueta, col_dc, col_an in [
    ("likes", "like_count", "likes"),
    ("comentarios", "comment_count", "comments"),
]:
    dc, an = df[col_dc].sum(), df[col_an].sum()
    print(f"{etiqueta:12s} Data API: {dc:>10,.0f} | Analytics: {an:>10,.0f} | dif: {dc - an:>8,.0f}")

== 6.1 Vídeos comparables: 371 de 375 ==
Vídeos con views = 0 en Analytics (sin % calculable): 1

== 6.2 Totales ==
Data API (view_count): 3,981,210
Analytics (views):     3,969,064
Diferencia: 12,146 (0.31%)

Vídeos según el signo de la diferencia:
diff_views
iguales                 228
Data API > Analytics    142
Data API < Analytics      1

== 6.3 Vídeos por tramo de diferencia relativa (valor absoluto) ==
diff_views_pct
exacta (0%)    227
hasta 1%       139
1% - 5%          2
más de 5%        2

Resumen de diff_views_pct (%):
count    370.00
mean       0.27
std        2.98
min       -0.38
25%        0.00
50%        0.00
75%        0.03
max       51.29

== 6.4 Mayores diferencias absolutas ==
   video_id                                                                                            title  is_public format              published_at  view_count    views  diff_views  diff_views_pct
6ImApUp7ifE                                      Previa BETIS-FEYENOORD: JUNTOS a por el 3 de

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18916\842505388.py:59: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(top[columnas].round(1).to_string(index=False))
